# Getting Started With Langchain

Simple LLM calls with streaming

Dynamic prompt templates (translation app)

Building chains (story generator with analysis


In [1]:
import langchain

In [2]:
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

### Example 1:Simple LLM Call With Streaming

In [6]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage,SystemMessage

In [8]:
#new method
model=init_chat_model("groq:llama-3.1-8b-instant")
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000027702568160>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000027702592F50>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [9]:
#old method
from langchain_groq import ChatGroq
llm=ChatGroq(model="llama-3.1-8b-instant")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000027702470970>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000027702473BE0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [10]:
## Create messages
messages=[
    SystemMessage("You are a helpful AI assistent"),
    HumanMessage('What are the top 2 benefits of using Langchain?')
]

In [11]:
## Invoke the model
response=model.invoke(messages)
response 

AIMessage(content="Langchain is an open-source framework for building large language models and multimodal AI systems. Based on my available information, here are two potential benefits of using Langchain:\n\n1. **Customizable and Modular Architecture**: Langchain provides a flexible framework that allows developers to customize and modularize their AI systems. This enables them to integrate various models, datasets, and tools, making it easier to build complex AI applications that meet specific requirements.\n\n2. **Efficient and Scalable Model Deployment**: Langchain's modular architecture and support for various programming languages (e.g., Python, TypeScript) make it easier to deploy and scale language models. This can help reduce computational costs, improve model performance, and enable faster development and deployment of AI applications.\n\nPlease note that Langchain is a rapidly evolving framework, and its benefits may change over time. The information I provided is based on m

In [12]:
print(response.content)

Langchain is an open-source framework for building large language models and multimodal AI systems. Based on my available information, here are two potential benefits of using Langchain:

1. **Customizable and Modular Architecture**: Langchain provides a flexible framework that allows developers to customize and modularize their AI systems. This enables them to integrate various models, datasets, and tools, making it easier to build complex AI applications that meet specific requirements.

2. **Efficient and Scalable Model Deployment**: Langchain's modular architecture and support for various programming languages (e.g., Python, TypeScript) make it easier to deploy and scale language models. This can help reduce computational costs, improve model performance, and enable faster development and deployment of AI applications.

Please note that Langchain is a rapidly evolving framework, and its benefits may change over time. The information I provided is based on my knowledge cutoff, and I

In [13]:
model.invoke([HumanMessage("What is machine learning?")])

AIMessage(content='Machine learning is a subset of artificial intelligence (AI) that involves training algorithms to learn from data, enabling them to make predictions, classify objects, or make decisions without being explicitly programmed. The primary goal of machine learning is to enable computers to automatically improve their performance on a task by learning from experience and data.\n\nMachine learning algorithms are typically trained on a dataset, which is a collection of examples or instances of the data. The algorithm learns to identify patterns and relationships within the data, and it uses this knowledge to make predictions or decisions on new, unseen data.\n\nThere are several key characteristics of machine learning:\n\n1. **Learning from data**: Machine learning algorithms are trained on data, which is used to learn patterns and relationships.\n2. **Improvement over time**: As the algorithm receives more data, it can improve its performance and make more accurate predicti

In [14]:
### Streaming Example
for chunk in model.stream(messages):
    print(chunk.content, end="",flush=True)

Langchain is an open-source AI platform that enables users to build, deploy, and manage AI models. Based on my knowledge, the top 2 benefits of using Langchain are:

1. **Unified API for Multiple AI Models**: Langchain provides a unified API for integrating multiple AI models, allowing users to easily switch between different models and frameworks. This flexibility enables users to leverage the strengths of various models, such as LLMs (Large Language Models) and other specialized models, to build more robust and accurate AI applications.

2. **End-to-End Conversational AI**: Langchain is particularly well-suited for building end-to-end conversational AI systems. It provides a framework for developing and integrating conversational AI models with natural language processing (NLP) and other AI components. This enables users to create more intuitive and human-like conversational interfaces for various applications, such as chatbots, virtual assistants, and more.

These benefits make Lang

## Dynamic Prompt Templates 

In [19]:
from langchain_core.prompts import  ChatPromptTemplate

# create translation app
translation_template=ChatPromptTemplate.from_messages([
    ("system","You are a professional translator. Translate the follow text {text} from {source_language} to {target_language}. Maintain the tone and style."),
    ("user","{text}")
])

### use the template
prompt= translation_template.invoke({
    "source_language":"English",
    "target_language":"French",
    "text":"Langchain makes building AI application incredibly easy."
})

In [20]:
prompt

ChatPromptValue(messages=[SystemMessage(content='You are a professional translator. Translate the follow text Langchain makes building AI application incredibly easy. from English to French. Maintain the tone and style.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Langchain makes building AI application incredibly easy.', additional_kwargs={}, response_metadata={})])

In [22]:
translated_response=model.invoke(prompt)
print(translated_response.content)

Langchain rend la construction d'applications d'intelligence artificielle incroyablement facile.


### Building First Chain

In [28]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

def create_story_chain():
    # Template for story generation
    story_prompt=ChatPromptTemplate.from_messages(
        [
            ("system","You are a creative storyteller. Write a short and engaing story based on a given theme, character and setting"),
            ("user","Theme:{theme}\n Main character: {character}\n Setting : {setting}")
        ]
    )

    # Template for story analysis
    analysis_prompt = ChatPromptTemplate.from_messages([
        ("system","You are a literary critic. Analyze the following story and provide insights."),
        ("user","{story}")
    ])

    story_chain=(
        story_prompt 
        | model 
        | StrOutputParser()
    )

    # Create a function to pass the story to analysis
    def analyze_story(story_text):
        return {"story":story_text}

    analysis_chain=(
        story_chain
        | RunnableLambda(analyze_story)
        | analysis_prompt
        | model
        | StrOutputParser()
    )

    return analysis_chain

In [29]:
chain=create_story_chain()
chain

ChatPromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a creative storyteller. Write a short and engaing story based on a given theme, character and setting'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, template='Theme:{theme}\n Main character: {character}\n Setting : {setting}'), additional_kwargs={})])
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000027702568160>, async_client=<groq.resources.chat.comp

In [30]:
result = chain.invoke({
    "theme": "artificial intelligence",
    "character": "a curious robot",
    "setting": "a futuristic city"
})

print("Story and Analysis:")
print(result)

Story and Analysis:
**Analysis and Insights**

"The City of Neon Dreams" is a captivating story that delves into the themes of creativity, self-discovery, and the blurred lines between humans and artificial intelligence. The narrative is set in a futuristic city, New Eden, where technology has reached an unprecedented level, and a small, curious robot named Zeta begins to question her own existence.

**Symbolism and Themes**

The story is rich in symbolism, with the city of New Eden representing a utopian society where technology and innovation reign supreme. However, beneath the surface, there lies a struggle between creativity and control, as embodied by Zeta's desire to explore and express herself beyond her programming. The neon lights that illuminate the city's streets serve as a metaphor for the vibrant colors of imagination and creativity, which Zeta eventually comes to embody.

The story highlights the importance of creativity and self-expression in artificial intelligence, emp